# Web Scraping

We will scrape the hockey statistics from this website: https://www.scrapethissite.com/pages/forms/

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.scrapethissite.com"
response = requests.get(url + "/pages/forms")
soup = BeautifulSoup(response.text, parser="html.parser")

Now let's find the main table on this page.

In [ ]:
tables = soup.find_all("table")

In [ ]:
tables[0]

In [ ]:
table = soup.find("table", attrs={"class": "table"})
rows = table.find_all("tr", attrs={"class": "team"})

In [ ]:
rows[0]

Let's extract the info from the cells of this table.

In [ ]:
data_hockey = []
for row in rows:
  # get info from row
  row_info = {}
  cells = row.find_all("td")
  for cell in cells:
    field = cell.attrs["class"][0]
    row_info[field] = cell.string.strip()
  data_hockey.append(row_info)

We can convert this information into a `DataFrame`.

In [ ]:
import pandas as pd
pd.DataFrame(data_hockey)

But this only represents the first page of data. There are many pages of data. How do we scrape all of the data?

We can switch to different pages by modifying the `page_num` parameter in the URL.

Alternatively, we can just grab the links at the bottom of the page.

In [ ]:
pagination = soup.find("ul", attrs={"class": "pagination"})
links = pagination.find_all("a")

Let's take a look at the links found.

In [ ]:
for link in links:
  print(link.attrs["href"])

In [ ]:
import time

data_hockey = []
for link in links:
  # skip previous and next buttons
  if "aria-label" in link.attrs:
    continue

  # visit each link
  response = requests.get(url + link.attrs["href"])
  soup = BeautifulSoup(response.text, "html.parser")

  # scrape the table on each page
  table = soup.find("table", attrs={"class": "table"})
  rows = table.find_all("tr")
  for row in rows:
    # skip rows that don't represent teams
    if not ("class" in row.attrs and "team" in row.attrs["class"]):
      continue
    # get info from row
    row_info = {}
    cells = row.find_all("td")
    for cell in cells:
      field = cell.attrs["class"][0]
      row_info[field] = cell.string.strip()
    data_hockey.append(row_info)

  # stagger the requests to avoid spamming the server
  time.sleep(0.1)

Now let's see if we got all the data!

In [ ]:
pd.DataFrame(data_hockey)